# 04 — Validation and extra experiments

Training-only validation for every selected value (d, m), then E4 training size, optional E5 random split, E6 regularised Mahalanobis.

Run notebooks 02 and 03 first: test accuracies at the selected values are read from their saved arrays.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import numpy as np

from src.config import OFFSETS, RESULTS_DIR
from src.data import load_coil20, make_split
from src.evaluate import CLASSIFIERS, lda_scorer, pca_scorer, select_by_validation

ARRAYS = RESULTS_DIR / "arrays"
names = list(CLASSIFIERS)

In [ ]:
X, y, pose = load_coil20()
pca = np.load(ARRAYS / "pca_acc.npz")
lda = np.load(ARRAYS / "lda_m_sweep.npz")

## Training-only validation (section 8.1)

For each split, leave-one-pose-out inside the 240 training images: 12 folds, each fitting on 220 images and validating on the 20 at the held-out pose. Test indices are never passed in. The selected value is then looked up on the test curve.

- Inside a fold N = 220, so PCA d ≤ 219, min-Mahal d ≤ N − c = 200, and LDA m ≤ 200. Candidates undefined in any fold are skipped.
- Ties go to the smallest d (or m).
- The held-out pose is 30° from its nearest remaining training pose, whereas test images are at most 15° away, so validation accuracy is pessimistic. It is used for ranking only.

### PCA: select d per classifier

In [ ]:
d_pca = [int(d) for d in pca["d_list"] if d <= 219]
pca_d = np.zeros((len(OFFSETS), len(names)), dtype=int)
pca_val = np.zeros(pca_d.shape)
pca_test = np.zeros(pca_d.shape)
for k, off in enumerate(OFFSETS):
    tr, _ = make_split(pose, off)
    for i, predict in enumerate(CLASSIFIERS.values()):
        d, scores = select_by_validation(X, y, pose, tr, d_pca, pca_scorer(predict, d_pca))
        pca_d[k, i], pca_val[k, i] = d, np.nanmax(scores)
        pca_test[k, i] = pca["acc"][k, i, np.flatnonzero(pca["d_list"] == d)[0]]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # all-nan min-Mahal columns above the cap
    best = np.nanmax(np.nanmean(pca["acc"], axis=0), axis=1)
for i, n in enumerate(names):
    print(f"{n:11s} d per split {pca_d[:, i]}  val {pca_val[:, i].mean():.4f}  "
          f"test {pca_test[:, i].mean():.4f}±{pca_test[:, i].std():.4f}  (best on test curve {best[i]:.4f})")

### PCA → LDA: select (m, d) jointly per classifier

m from the sweep in notebook 03, excluding 220 (above the fold cap of 200).

In [ ]:
m_list = [int(m) for m in lda["m_list"] if m <= 200]
d_lda = [int(d) for d in lda["d_list"]]
lda_m = np.zeros((len(OFFSETS), len(names)), dtype=int)
lda_d = np.zeros_like(lda_m)
lda_val = np.zeros(lda_m.shape)
lda_test = np.zeros(lda_m.shape)
for k, off in enumerate(OFFSETS):
    tr, _ = make_split(pose, off)
    for i, predict in enumerate(CLASSIFIERS.values()):
        candidates, score_fn = lda_scorer(predict, m_list, d_lda)
        (mp, d), scores = select_by_validation(X, y, pose, tr, candidates, score_fn)
        lda_m[k, i], lda_d[k, i], lda_val[k, i] = mp, d, np.nanmax(scores)
        a = np.flatnonzero(lda["m_list"] == mp)[0]
        lda_test[k, i] = lda["acc"][a, k, i, np.flatnonzero(lda["d_list"] == d)[0]]
for i, n in enumerate(names):
    print(f"{n:11s} m {lda_m[:, i]}  d {lda_d[:, i]}  val {lda_val[:, i].mean():.4f}  "
          f"test {lda_test[:, i].mean():.4f}±{lda_test[:, i].std():.4f}")

In [ ]:
np.savez(ARRAYS / "selected.npz", classifiers=names,
         pca_d=pca_d, pca_val=pca_val, pca_test=pca_test,
         lda_m=lda_m, lda_d=lda_d, lda_val=lda_val, lda_test=lda_test)

## E4: training size

step = 4, 6, 9, 12 → 18, 12, 8, 6 training images per class (pose spacing 20°, 30°, 45°, 60°); `step` splits each. For every split, d (PCA) and (m, d) (PCA → LDA) are selected per classifier by the same leave-one-pose-out validation, then scored on the test set. All caps (N − 1, N − c) follow from each training set.

In [ ]:
from src.evaluate import run_trainsize_experiment

steps = [4, 6, 9, 12]
d_cand = sorted(set(range(1, 51)) | set(range(55, 240, 5)))
m_cand = [20, 30, 50, 75, 100, 150, 200]
e4 = run_trainsize_experiment(X, y, pose, steps, d_cand, m_cand)
np.savez(ARRAYS / "trainsize_acc.npz", steps=steps, classifiers=names,
         **{f"{key}_{s}": v for s, r in e4.items() for key, v in r.items()})
for s in steps:
    r = e4[s]
    print(f"{72 // s:2d}/class  PCA: " + "  ".join(f"{n} {r['pca_test'][:, i].mean():.4f}±{r['pca_test'][:, i].std():.4f}" for i, n in enumerate(names)))
    print(f"          LDA: " + "  ".join(f"{n} {r['lda_test'][:, i].mean():.4f}±{r['lda_test'][:, i].std():.4f}" for i, n in enumerate(names)))
    print(f"          median selected PCA d {np.median(r['pca_d'], axis=0)}, LDA m {np.median(r['lda_m'], axis=0)}")

## E5 (optional): random vs pose-spaced split

Same training size, training poses drawn at random per class instead of evenly spaced; 6 seeds each. Compared at 12 per class (step 6) and 36 per class (step 2).

Random sampling leaves uneven pose coverage: some objects get gaps of 50–110° between training poses, while the pose-spaced split keeps every test image within (step × 5°)/2 of a training pose. So the pose-spaced split gives controlled, comparable coverage across objects and splits.

In [ ]:
from src.config import SEED
from src.data import make_random_split
from src.evaluate import run_pca_experiment

e5 = {}
for n in (12, 36):
    rand_splits = [make_random_split(y, n, SEED + k) for k in OFFSETS]
    e5[f"random_{n}"] = run_pca_experiment(X, y, pose, pca["d_list"], splits=rand_splits)
    e5[f"spaced_{n}"] = run_pca_experiment(X, y, pose, pca["d_list"], step=72 // n)
np.savez(ARRAYS / "random_split_acc.npz", d_list=pca["d_list"], classifiers=names, **e5)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    for n in (12, 36):
        mr, ms = np.nanmean(e5[f"random_{n}"], axis=0), np.nanmean(e5[f"spaced_{n}"], axis=0)
        for d in (5, 20, 50, 100):
            j = np.flatnonzero(pca["d_list"] == d)[0]
            print(f"{n}/class d={d:3d}  " + "  ".join(f"{c} random {mr[i, j]:.4f} / spaced {ms[i, j]:.4f}" for i, c in enumerate(names)))

tr, _ = make_random_split(y, 12, SEED)
gaps = [5 * np.diff(np.r_[p, p[0] + 72]).max() for p in (np.sort(pose[tr][y[tr] == k]) for k in range(20))]
print(f"random 12/class, seed {SEED}: largest gap between training poses per object {min(gaps)}–{max(gaps)}° (spaced: 30°)")

## E6 (optional): regularised Mahalanobis

$\Sigma_{reg} = \Sigma_w + \epsilon \cdot (\mathrm{tr}\,\Sigma_w / d)\, I$ (slide 93), in PCA space. With ε > 0 the cap d ≤ N − c no longer applies. Test curves for each ε, then (ε, d) selected jointly by training-only validation.

In [ ]:
from functools import partial

from src.classifiers import min_mahal_predict

eps_list = [0, 1e-4, 1e-3, 1e-2, 1e-1]
reg = {f"eps={e:g}": partial(min_mahal_predict, eps=e) for e in eps_list}
acc_reg = run_pca_experiment(X, y, pose, pca["d_list"], classifiers=reg)
np.savez(ARRAYS / "reg_mahal_acc.npz", acc=acc_reg, d_list=pca["d_list"], eps_list=eps_list)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    mreg = np.nanmean(acc_reg, axis=0)
for d in (20, 50, 100, 200, 220, 239):
    j = np.flatnonzero(pca["d_list"] == d)[0]
    print(f"d={d:3d}  " + "  ".join(f"{n} {mreg[i, j]:.4f}" for i, n in enumerate(reg)))

In [ ]:
d_val = [int(d) for d in pca["d_list"] if d <= 219]
reg_eps = np.zeros(len(OFFSETS))
reg_d = np.zeros(len(OFFSETS), dtype=int)
reg_test = np.zeros(len(OFFSETS))
for k, off in enumerate(OFFSETS):
    tr, _ = make_split(pose, off)
    best = (-1.0, None, None)
    for e in eps_list:   # joint (eps, d) argmax; ties keep the smaller eps
        d, scores = select_by_validation(X, y, pose, tr, d_val, pca_scorer(partial(min_mahal_predict, eps=e), d_val))
        if np.nanmax(scores) > best[0]:
            best = (np.nanmax(scores), e, d)
    _, reg_eps[k], reg_d[k] = best
    reg_test[k] = acc_reg[k, eps_list.index(reg_eps[k]), np.flatnonzero(pca["d_list"] == reg_d[k])[0]]
np.savez(ARRAYS / "reg_mahal_selected.npz", eps=reg_eps, d=reg_d, test=reg_test)
print("selected eps", reg_eps, "d", reg_d)
print(f"test {reg_test.mean():.4f}±{reg_test.std():.4f}  (unregularised, validated: {pca_test[:, 2].mean():.4f})")

### Quick look (not a report figure)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
per_class = [72 // s for s in steps]
for i, n in enumerate(names):
    ax1.plot(per_class, [1 - e4[s]["pca_test"][:, i].mean() for s in steps], marker="o", label=f"PCA {n}")
    ax1.plot(per_class, [1 - e4[s]["lda_test"][:, i].mean() for s in steps], marker="s", ls="--", label=f"LDA {n}")
ax1.set(xlabel="training images per class", ylabel="error rate", title="E4: training size (validated d, m)")
ax1.legend(fontsize=8)
for i, n in enumerate(reg):
    ax2.plot(pca["d_list"], 1 - mreg[i], label=n)
ax2.set(xscale="log", ylim=(0, 0.5), xlabel="d", ylabel="error rate", title="E6: regularised min-Mahal")
ax2.legend(fontsize=8)
plt.show()